# IOAI — 2025 Team Selection Day4 Nlp Masked Word (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day4-nlp-masked-word'
for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','public_test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 4 — Masked-Word Position (Kazakhstan IOAI Team Selection 2025)

> **Kazakhstan – Team Selection 2025 · Day 4 (NLP / Transformers)**

One word was deleted from a Kazakh sentence; given the **masked sentence**, predict the **0-based index** of the removed word in the original. Score = **accuracy** over a held-out validation split. This baseline uses a simple **frequency prior + capitalisation heuristic** (if the sentence starts lower-case, the leading capitalised word was likely removed → index 0). Writes `submission_val.csv` (graded) and `submission.csv` (Kaggle).

*Real-data notice:* original competition data from the public up-solving mirror `kaggle.com/competitions/tst-day-4-upsolving`. Validation is built by masking a random word from held-out training sentences (true indices known).

In [ ]:
import os, pandas as pd, numpy as np, random
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day4_NLP_Masked_Word"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."
DATA=_find("train.csv")
train=pd.read_csv(os.path.join(DATA,"train.csv"))            # col: sentence (full)
val  =pd.read_csv(os.path.join(DATA,"val.csv"))             # ID, masked_sentence  (graded by accuracy)
test =pd.read_csv(os.path.join(DATA,"public_test.csv"))     # ID, masked_sentence  (Kaggle)
print("train",train.shape,"val",val.shape,"test",test.shape)


## Frequency prior + capitalisation heuristic

In [ ]:
random.seed(0)
# build (masked, idx) pairs from training sentences to estimate the index prior
prior={}
for s in train["sentence"].dropna().astype(str):
    w=s.split()
    if not (2<=len(w)<=25): continue
    j=random.randrange(len(w))
    prior[j]=prior.get(j,0)+1
mode_idx=max(prior, key=prior.get)
def guess(masked):
    w=str(masked).split()
    if w and w[0][:1].islower():   # sentence starts lowercase -> leading capitalised word was removed
        return 0
    return min(mode_idx, len(w))   # otherwise fall back to the most common index
for df,out in [(val,"submission_val.csv"),(test,"submission.csv")]:
    pd.DataFrame({"ID":df["ID"], "word_index":[guess(m) for m in df["masked_sentence"]]}).to_csv(out,index=False)
print("baseline mode index =", mode_idx, "| wrote submissions")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)